# Newton's Method Runs Away

**Topic:** Root finding &nbsp;·&nbsp; **Difficulty:** Hard

Newton's method converges quadratically (when it converges). Started in the wrong
place on this function, the tangent line points off toward infinity and the
iterate never comes back.

The industrial fix is a safeguard: keep a bracket you know contains a root, take
the Newton step when it lands inside the bracket, and fall back to bisection when
it does not. You get Newton's speed with bisection's guarantee.

## The data (created in the next cell)

- `g` and `dg`: a scalar function with a root in `[0.2, 4.0]`, and its derivative.
- `g` has a nearly flat region where the plain Newton step overshoots badly.
- Starting point `x0 = 3.9`, bracket `[0.2, 4.0]`.

## Your task

With **NumPy only**:

1. Implement safeguarded Newton on the bracket `[lo, hi] = [0.2, 4.0]`:
   at each step, propose the Newton step `x - g(x)/dg(x)`. If it falls outside
   the current bracket or the derivative is tiny, use the bisection midpoint
   instead. Shrink the bracket using the sign of `g` each iteration.
2. Stop when `|g(x)| < 1e-12` or after 200 iterations.
3. Also run **plain** Newton from the same `x0` for 200 iterations.
4. Set `answers` to a dict with `"root"` (the safeguarded result),
   `"n_iters"` (int, safeguarded iterations used), and `"plain_diverged"`
   (1.0 if plain Newton ended more than 1.0 away from the root or non-finite,
   else 0.0).

## Evaluation

The root is checked to a relative tolerance of 1e-6 and `|g(root)|` must be below 1e-10 (which pins the root to about 3e-11).

In [ ]:
import numpy as np

def g(x):
    return np.arctan(4.0 * (x - 1.1)) - 0.35

def dg(x):
    return 4.0 / (1.0 + (4.0 * (x - 1.1)) ** 2)

lo, hi, x0 = 0.2, 4.0, 3.9
print(f"g(lo) = {g(lo):.4f}, g(hi) = {g(hi):.4f}   (opposite signs: bracket is valid)")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use g, dg, lo, hi and x0 from the cell above, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("root", [], 1.1912571237076062, -0.19946172792723182, 1e-06),
    ("plain_diverged", [], 1.0, -0.16743801481449916, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "root" in answers:
    _gv = float(abs(g(float(answers["root"]))))
    print(f"  {'|g(root)| < 1e-10':<24} {'ok' if _gv < 1e-10 else 'wrong'}  (|g| = {_gv:.2e})")
    _ok = _ok and _gv < 1e-10

print("=" * 56)
if _ok:
    print("PASSED. Nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

`arctan` flattens out, so far from the root the derivative is tiny and the Newton
step is enormous. From 3.9 it jumps past the bracket entirely and oscillates
outward. The safeguard notices the proposal left `[lo, hi]`, takes a bisection
step instead, and keeps halving until the iterate is close enough that Newton's
quadratic convergence takes over.

```python
import numpy as np

a, b = lo, hi
x = x0
n_iters = 0
for i in range(200):
    n_iters = i + 1
    gx = g(x)
    if abs(gx) < 1e-12:
        break
    d = dg(x)
    step = x - gx / d if abs(d) > 1e-14 else np.inf
    if not np.isfinite(step) or not (a < step < b):
        step = 0.5 * (a + b)
    x = step
    if g(a) * g(x) <= 0:
        b = x
    else:
        a = x

root = float(x)

z = x0
for _ in range(200):
    d = dg(z)
    if abs(d) < 1e-300 or not np.isfinite(z):
        break
    z = z - g(z) / d

diverged = 1.0 if (not np.isfinite(z) or abs(z - root) > 1.0) else 0.0

answers = {"root": root, "n_iters": int(n_iters), "plain_diverged": diverged}
```